In [15]:
import json
import logging

from torch.utils.data import DataLoader, Dataset
from sentence_transformers import (
    InputExample,
    LoggingHandler,
    SentenceTransformer,
    losses,
)

In [2]:
logging.basicConfig(
    format="%(asctime)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
    handlers=[LoggingHandler()],
)

In [ ]:
class MSMARCODataset(Dataset):
    def __init__(self, dataset_path: str):
        with open(dataset_path, "r") as f:
            self.dataset = [json.loads(x) for x in f]

    def __getitem__(self, idx):
        item = self.dataset[idx]
        return InputExample(texts=[item["query"], item["positive"], item["negative"]])

    def __len__(self):
        return len(self.dataset)

In [3]:
train_dataset = MSMARCODataset("../data/marco/triplet.json")
train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=64)

In [ ]:
model = SentenceTransformer("msmarco-distilbert-base-v4-v1")
model.max_seq_length = 300

In [ ]:
train_loss = losses.MultipleNegativesRankingLoss(model=model)

In [ ]:
model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    use_amp=True,
    epochs=10,
    warmup_steps=10000,
    optimizer_params={"lr": 2e-5},
)

In [ ]:
model.save("data/models/test")